In [8]:
import numpy as np
import pandas as pd

In [2]:
input_dir = "../data/input"

review_df = pd.read_parquet(f"{input_dir}/merged_df.parquet")
review_df = review_df[review_df["verified_purchase"] == True][[
    "review_id", "asin", "parent_asin", "user_id",
    "review_datetime",
    # "review_title", "review_text", "review_images",
    "helpful_vote", "review_rating"
]]

user_df = pd.read_parquet(f"{input_dir}/user_df.parquet")


In [6]:
user_df

,user_id,main_category,category,num_review_given,avg_rating_given,latest_user_datetime
0,ae22232ob6s7uac75jufyrgbshiq,appstore for android,nan,1,5.000000,2020-05-16
1,ae22236afrrsmqikgg7tptb75qea,appstore for android,nan,6,4.166667,2018-12-15
2,ae2224fsuk5av5r2usyxinuntw7q,appstore for android,nan,3,2.333333,2015-03-24
3,ae2226penzttcdkfgrtucux2nu2q,appstore for android,nan,2,3.000000,2021-05-13
4,ae2227qsjsf6hv3xttigqapa6whq,appstore for android,nan,11,3.454545,2020-08-22
...,...,...,...,...,...,...
2330390,ahzzzipdm2xjez4rz7bns4yhccka,appstore for android,nan,1,4.000000,2013-02-27
2330391,ahzzzngyuwv64yjbzgd3uc5y3liq,appstore for android,nan,2,5.000000,2015-01-15
2330392,ahzzzuba3pttfi67ios2mftjavea,appstore for android,nan,1,5.000000,2014-12-31
2330393,ahzzzy4dflawpbqyfqfwvacngura,appstore for android,nan,4,4.500000,2016-08-19


In [ ]:
review_df["review_datetime"] = pd.to_datetime(review_df["review_datetime"])

REFERENCE_DATE = review_df["review_datetime"].max()
modified_review_df = review_df.copy()

# Recency weight (exponential decay)
modified_review_df["age_days"] = (REFERENCE_DATE - modified_review_df["review_datetime"]).dt.days
modified_review_df["recency_weight"] = np.exp(-modified_review_df["age_days"] / 365)

# Helpful-vote weight (log-scaled)
modified_review_df["helpful_weight"] = 1 + np.log1p(modified_review_df["helpful_vote"])

# Confidence Weight
modified_review_df["confidence"] = modified_review_df["recency_weight"] * modified_review_df["helpful_weight"]

# Modified Review Rating
modified_review_df["weighted_review_rating"] = modified_review_df["review_rating"] * modified_review_df["confidence"]

modified_review_df

c:\Users\user\OneDrive\Desktop\Recommender\Recommender\venv\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,review_id,asin,parent_asin,user_id,review_datetime,helpful_vote,review_rating,age_days,recency_weight,helpful_weight,confidence,weighted_review_rating
3,1,b00ctq6sig,b00ctq6sig,ahspldnw5oouk2plh7gxlacfbznq,2015-02-16,0,5.0,3129,0.000189,1.000000,0.000189,0.000946
4,2,b0066wjlu6,b0066wjlu6,ahspldnw5oouk2plh7gxlacfbznq,2013-03-04,0,5.0,3843,0.000027,1.000000,0.000027,0.000134
5,3,b00kcymawk,b00kcymawk,ah6catodivpvuojewhrsrcskaoha,2019-06-20,0,4.0,1544,0.014550,1.000000,0.014550,0.058202
6,4,b00p1rk566,b00p1rk566,aeiny4xoinmmjck5gz3m6mmhbn6a,2014-12-11,0,4.0,3196,0.000157,1.000000,0.000157,0.000630
7,5,b00cwy76cc,b00cwy76cc,aeiny4xoinmmjck5gz3m6mmhbn6a,2013-07-28,0,4.0,3697,0.000040,1.000000,0.000040,0.000160
...,...,...,...,...,...,...,...,...,...,...,...,...
4459897,4880175,b07th23gwv,b07th23gwv,afxn4h3e55alrifjpsiqfyoxklha,2022-03-26,2,5.0,534,0.231537,2.098612,0.485907,2.429537
4459898,4880176,b0763n3mvw,b0763n3mvw,agokecumai3w6mc7a7kbfbjvpn7q,2020-12-25,0,5.0,990,0.066382,1.000000,0.066382,0.331910
4459899,4880177,b00et56y48,b00et56y48,ah6amundosz2sgdidmysre4rnmdq,2021-01-06,1,1.0,978,0.068601,1.693147,0.116151,0.116151
4459900,4880178,b009zkspdk,b009zkspdk,ah4pj73qn75ajm5vsct53aoadcga,2013-05-28,2,5.0,3758,0.000034,2.098612,0.000071,0.000354


In [5]:
review_df["review_rating"].unique()

array([5., 4., 1., 3., 2.])